# Land Surface Temperature from Landsat — Tel Aviv

Uses **Landsat Collection 2 Level-2** (Landsat 8 + 9), which includes pre-computed LST (`ST_B10`) processed by USGS.

| Item | Detail |
|---|---|
| Source | `LANDSAT/LC08/C02/T1_L2` + `LANDSAT/LC09/C02/T1_L2` |
| LST band | `ST_B10` — scaled to °C by USGS |
| Resolution | 30 m |
| Period | Last 12 months |
| Outputs | `tel_aviv_lst_median.tif` + `tel_aviv_lst_p90.tif` in Google Drive `GEE_Exports/` |

**Run in:** Google Colab

---

## How LST is Calculated

Landsat Band 10 is a **thermal infrared sensor** — it measures heat radiation emitted by the Earth's surface (not reflected sunlight). The processing chain:

1. **Raw DN → Kelvin:** multiply by `0.00341802`, add `149.0` (USGS calibration formula).
2. **Kelvin → Celsius:** subtract `273.15`.
3. USGS also applies **atmospheric correction** (removes atmosphere heat interference) and **emissivity correction** (concrete, water, and vegetation emit heat differently). Both are baked into the pre-computed `ST_B10` band — you only need to apply the scale factor.

### Why median and 90th percentile?

- **Median** — representative typical temperature, robust to cloud remnants or sensor noise in individual scenes.
- **90th percentile (P90)** — captures genuinely hot conditions (heat waves, summer peaks) without being skewed by a single outlier pixel or bad scene.

In [ ]:
!pip install geemap -q

In [ ]:
import ee
import geemap
import datetime

ee.Authenticate()
ee.Initialize(project='your-gee-project-id')  # <-- replace with your GEE project ID

In [ ]:
# ── Study area ───────────────────────────────────────────────────────────────
tel_aviv = ee.Geometry.BBox(34.739, 32.030, 34.840, 32.130)

# ── Date range: last 12 months ───────────────────────────────────────────────
end_date   = datetime.date.today().isoformat()
start_date = (datetime.date.today() - datetime.timedelta(days=365)).isoformat()
print(f'Period: {start_date} → {end_date}')

In [ ]:
# ── Scale + cloud-mask helpers ───────────────────────────────────────────────
LST_SCALE  = 0.00341802
LST_OFFSET = 149.0 - 273.15  # combined offset: raw DN → °C directly

def apply_lst_scale(image):
    lst = (image.select('ST_B10')
               .multiply(LST_SCALE)
               .add(LST_OFFSET)
               .rename('LST'))
    return image.addBands(lst)

def mask_landsat_clouds(image):
    qa = image.select('QA_PIXEL')
    # Bit 3 = cloud shadow, Bit 4 = cloud
    cloud_mask = (qa.bitwiseAnd(1 << 3).eq(0)
                   .And(qa.bitwiseAnd(1 << 4).eq(0)))
    return image.updateMask(cloud_mask)

In [ ]:
# ── Load Landsat 8 + 9 Collection 2 Level-2 ─────────────────────────────────
def load_landsat(collection_id):
    return (ee.ImageCollection(collection_id)
              .filterBounds(tel_aviv)
              .filterDate(start_date, end_date)
              .filter(ee.Filter.lt('CLOUD_COVER', 15))
              .map(mask_landsat_clouds)
              .map(apply_lst_scale)
              .select('LST'))

l8 = load_landsat('LANDSAT/LC08/C02/T1_L2')
l9 = load_landsat('LANDSAT/LC09/C02/T1_L2')
collection = l8.merge(l9)

n = collection.size().getInfo()
print(f'Landsat 8 scenes : {l8.size().getInfo()}')
print(f'Landsat 9 scenes : {l9.size().getInfo()}')
print(f'Total scenes     : {n}')
print(f'(~{n * 16 // 365} scenes/month expected at 16-day revisit)')

In [ ]:
# ── Build composites ─────────────────────────────────────────────────────────

# Median: typical temperature across the year
lst_median = collection.median().clip(tel_aviv).rename('LST')

# 90th percentile: hot-peak temperature, outlier-resistant
# Uses ee.Reducer.percentile — each pixel gets the 90th-highest value
# across all cloud-free scenes, ignoring extreme single-scene spikes.
lst_p90 = collection.reduce(ee.Reducer.percentile([90])).clip(tel_aviv).rename('LST')

print('Composites built.')

In [ ]:
# ── Statistics ───────────────────────────────────────────────────────────────
def print_stats(image, label):
    stats = image.reduceRegion(
        reducer=ee.Reducer.minMax().combine(ee.Reducer.mean(), sharedInputs=True),
        geometry=tel_aviv,
        scale=30,
        maxPixels=1e9
    ).getInfo()
    print(f"{label}  →  min {stats['LST_min']:.1f}°C  |  mean {stats['LST_mean']:.1f}°C  |  max {stats['LST_max']:.1f}°C")

print_stats(lst_median, 'Median')
print_stats(lst_p90,    'P90   ')

In [ ]:
# ── Interactive map preview ──────────────────────────────────────────────────
Map = geemap.Map(center=[32.08, 34.79], zoom=12)

lst_vis = {
    'min': 20, 'max': 50,
    'palette': ['#313695', '#4575b4', '#74add1', '#abd9e9',
                '#fee090', '#fdae61', '#f46d43', '#d73027', '#a50026']
}

Map.addLayer(lst_median, lst_vis, 'LST Median (°C)')
Map.addLayer(lst_p90,    lst_vis, 'LST P90 (°C)', shown=False)
Map.add_colorbar(lst_vis, label='LST (°C)')
Map

In [ ]:
# ── Export both layers to Google Drive ──────────────────────────────────────
def export_lst(image, name):
    task = ee.batch.Export.image.toDrive(
        image=image,
        description=name,
        folder='GEE_Exports',
        fileNamePrefix=name,
        region=tel_aviv,
        scale=30,
        crs='EPSG:4326',
        maxPixels=1e9,
        fileFormat='GeoTIFF'
    )
    task.start()
    return task

task_median = export_lst(lst_median, 'tel_aviv_lst_median')
task_p90    = export_lst(lst_p90,    'tel_aviv_lst_p90')

print('Both export tasks started.')
print('Output → Google Drive/GEE_Exports/tel_aviv_lst_median.tif')
print('Output → Google Drive/GEE_Exports/tel_aviv_lst_p90.tif')

In [ ]:
# ── (Optional) Monitor both tasks ───────────────────────────────────────────
import time

tasks = {'Median': task_median, 'P90': task_p90}

while any(t.active() for t in tasks.values()):
    for label, task in tasks.items():
        print(f"{label}: {task.status()['state']}")
    print('---')
    time.sleep(30)

for label, task in tasks.items():
    print(f"{label} final state: {task.status()['state']}")

## After the Export Finishes

1. Download both `.tif` files from `Google Drive/GEE_Exports/`.
2. Place them in `Layers/` in this project.
3. Load in subsequent notebooks:

```python
import rasterio

with rasterio.open('Layers/tel_aviv_lst_median.tif') as src:
    lst_median = src.read(1)   # °C, shape (rows, cols)

with rasterio.open('Layers/tel_aviv_lst_p90.tif') as src:
    lst_p90 = src.read(1)      # °C, shape (rows, cols)
```

## Notes

- Landsat revisit is ~16 days → ~22 scenes/year over Tel Aviv before cloud filtering.
- LST is measured at ~10:30 local time (daytime overpass) — it reflects surface heat, not air temperature.
- NDVI (10 m) and LST (30 m) have different resolutions — both get aggregated to the 100 m analysis grid in the next step.